# Cookie Cats A/B test — should the first gate move from level 30 to level 40?

**Context.** In the mobile puzzle game *Cookie Cats*, players hit a *gate* where they must wait or pay to continue.
The team tested moving the first gate from level 30 (**control**) to level 40 (**treatment**) on 90,189 new players.

**Decision to support:** ship gate 40 or keep gate 30?
**Primary metric:** 7-day retention (long-term engagement). **Secondary:** 1-day retention. **Guardrail:** game rounds played.

Plan: (1) data and randomisation checks → (2) frequentist tests with multiple-testing correction → (3) Bayesian view →
(4) guardrail → (5) robustness to the problems found → (6) recommendation and follow-up test design.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
import abtest
df = pd.read_csv('../data/raw/cookie_cats.csv')
print(df.shape, '| unique users:', df.userid.is_unique, '| missing values:', int(df.isna().sum().sum()))
df.head()

(90189, 5) | unique users: True | missing values: 0


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## 1. Health checks
### Sample ratio mismatch (SRM)
Players were meant to be split 50/50. A chi-square test checks whether the observed split is plausible.

In [2]:
ctrl, trt = df[df.version == 'gate_30'], df[df.version == 'gate_40']
abtest.srm_check(len(ctrl), len(trt))

{'n_control': 44700,
 'n_treatment': 45489,
 'share_treatment': 0.5043741476233243,
 'chi2': 6.9024049496058275,
 'p_value': 0.008607987810836262}

**p = 0.009 → SRM detected.** Gate 40 has 789 more players than expected. In industry practice an SRM below p < 0.01
is a red flag: the assignment or logging may be broken, so results need a robustness check (section 5) before anyone acts on them.

Where does the extra traffic sit? (Descriptive only — rounds played is itself affected by the treatment.)

In [3]:
band = pd.cut(df.sum_gamerounds, [-1, 29, 39, np.inf], labels=['0-29 (never reached a gate)', '30-39', '40+'])
pd.crosstab(band, df.version).assign(extra_in_gate_40=lambda t: t.gate_40 - t.gate_30)

version,gate_30,gate_40,extra_in_gate_40
sum_gamerounds,,,
0-29 (never reached a gate),28044,28876,832
30-39,3090,2786,-304
40+,13566,13827,261


The excess (+832) is concentrated in players who **never reached level 30**, i.e. who never saw either gate. These players have very low retention, so if they are spurious they would drag the treatment's numbers down.

### Outliers in engagement

In [4]:
df.sum_gamerounds.describe(percentiles=[.5, .9, .99, .999]).round(1)

count    90189.0
mean        51.9
std        195.1
min          0.0
50%         16.0
90%        134.0
99%        493.0
99.9%     1073.6
max      49854.0
Name: sum_gamerounds, dtype: float64

One player logged 49,854 rounds (≈ 20× the next highest). Retention metrics are binary so they are unaffected, but for the guardrail metric I winsorise at the 99.9th percentile.

## 2. Primary and secondary metrics — two-proportion z-tests

In [5]:
tests = {}
for m in ['retention_1', 'retention_7']:
    tests[m] = abtest.two_proportion_ztest(ctrl[m].sum(), len(ctrl), trt[m].sum(), len(trt))
pd.DataFrame(tests).T[['rate_control', 'rate_treatment', 'abs_diff', 'rel_diff', 'ci_low', 'ci_high', 'p_value']].astype(float).round(4)

,rate_control,rate_treatment,abs_diff,rel_diff,ci_low,ci_high,p_value
retention_1,0.4482,0.4423,-0.0059,-0.0132,-0.0124,0.0006,0.0744
retention_7,0.1902,0.1820,-0.0082,-0.0431,-0.0133,-0.0031,0.0016


Two metrics are tested, so I apply a **Holm-Bonferroni** correction to keep the family-wise error rate at 5%.

In [6]:
pd.DataFrame(abtest.holm_correction({m: t['p_value'] for m, t in tests.items()})).T

,p_value,p_adjusted,significant
retention_7,0.001554,0.003108,True
retention_1,0.07441,0.07441,False


![retention](../images/retention.png)

**7-day retention drops by 0.82 pp (19.0% → 18.2%, −4.3% relative), significant after correction.** 1-day retention also points down (−0.59 pp) but is not significant.

## 3. Bayesian view — what the business actually asks: *how likely is gate 40 to be better, and what do we risk?*

In [7]:
bayes = {m: abtest.bayesian_beta_binomial(ctrl[m].sum(), len(ctrl), trt[m].sum(), len(trt)) for m in tests}
pd.DataFrame(bayes).T.round(5)

,prob_treatment_better,expected_loss_ship_treatment,expected_loss_ship_control,lift_ci_low,lift_ci_high
retention_1,0.03768,0.00595,0.00005,-0.01238,0.00060
retention_7,0.00076,0.00819,0.00000,-0.01328,-0.00309


![posterior](../images/posterior_r7.png)

P(gate 40 has better 7-day retention) ≈ **0.08%**. Expected loss of shipping gate 40 ≈ 0.82 pp vs ≈ 0 for keeping gate 30.

## 4. Guardrail — rounds played (bootstrap, winsorised)

In [8]:
cap = df.sum_gamerounds.quantile(0.999)
bs = abtest.bootstrap_diff(ctrl.sum_gamerounds.clip(upper=cap).values, trt.sum_gamerounds.clip(upper=cap).values, n_boot=3000)
{k: round(v, 3) for k, v in bs.items() if k != 'samples'}

{'diff': -0.096, 'ci_low': -1.305, 'ci_high': 1.226, 'prob_negative': 0.557}

No detectable difference in rounds played (95% CI includes 0). The gate changes *who comes back*, not how much returning players play in their first two weeks.

## 5. Robustness — does the conclusion survive the SRM?
Worst-case scenario: the 789 extra treatment players are spurious, low-engagement, non-retained users. Remove them and re-test.

In [9]:
excess = len(trt) - len(ctrl)
low = trt[(trt.sum_gamerounds < 30) & (~trt.retention_7)]
trt_wc = trt.drop(low.sample(excess, random_state=1).index)
abtest.two_proportion_ztest(ctrl.retention_7.sum(), len(ctrl), trt_wc.retention_7.sum(), len(trt_wc))

{'rate_control': np.float64(0.19020134228187918),
 'rate_treatment': np.float64(0.18521252796420581),
 'abs_diff': np.float64(-0.004988814317673368),
 'rel_diff': np.float64(-0.026229122559397737),
 'z': -1.9100272868381478,
 'p_value': 0.056129700085283535,
 'ci_low': np.float64(-0.010107954210090041),
 'ci_high': np.float64(0.00013032557474330648)}

![robustness](../images/robustness.png)

Even in the worst case the effect stays negative (−0.50 pp), but it is no longer significant (p = 0.056). **The direction is robust; the size is uncertain.**

## 6. Design check and follow-up

In [10]:
base = tests['retention_7']['rate_control']
print(f"MDE of this test (80% power): {abtest.minimum_detectable_effect(base, len(ctrl))*100:.2f} pp")
print(f"Players per group to detect a 0.5 pp drop: {abtest.sample_size_per_group(base, -0.005):,}")

MDE of this test (80% power): 0.74 pp
Players per group to detect a 0.5 pp drop: 95,737


![power](../images/power_curve.png)

## Recommendation

1. **Keep the gate at level 30.** Every analysis points the same way (frequentist, Bayesian, worst-case SRM) and the
   asymmetric risk favours the status quo: ≈ 0.8 fewer retained players per 100 new installs with gate 40, no upside on engagement.
2. **Fix the assignment bug before the next experiment.** The SRM is concentrated in players who never reached level 30 —
   check bot traffic, duplicate installs and event logging for the gate_40 bucket.
3. **If product still wants gate 40**, run a confirmation test sized for a 0.5 pp effect: **≈ 96K players per group**, with
   an SRM alert on day 1.

**Limitations:** no timestamps (cannot check novelty effects or peeking), no revenue data (gates also drive in-app purchases),
single game and period.